# Time Series Prediction with PyTorch

In [ ]:
import random

import lightning.pytorch as pl
import pandas as pd
import ray
from matplotlib import pyplot as plt
from pytorch_forecasting import DeepAR, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.examples import generate_ar_data
from pytorch_forecasting.metrics import NormalDistributionLoss

Initialize Ray

---

In [ ]:
if ray.is_initialized():
    ray.shutdown()

display(ray.init(address="auto"))
display(ray.cluster_resources())

Create training dataset using `TimeSeriesDataSet`.

---

In [ ]:
data = generate_ar_data(seasonality=10.0, timesteps=400, n_series=100, seed=42)
data["static"] = 2
data["date"] = pd.Timestamp("2020-01-01") + pd.to_timedelta(data.time_idx, "D")

display(data.head())

In [ ]:
plt.figure(figsize=(10, 6))

for series_id in data.series.unique()[:10]:
    series_data = data[data.series == series_id]
    plt.plot(
        series_data.date,
        series_data.value,
        label=f"Series {series_id}",
    )

plt.legend()
plt.title("Generated Time Series Data")
plt.xlabel("Date")
plt.ylabel("Value")
plt.show()

Using the training dataset, create a validation dataset with `from_dataset()`. Similarly, a test dataset or later a dataset for inference can be created. You can store the dataset parameters directly if you do not wish to load the entire training dataset at inference time.

---

In [ ]:
# Ensure categorical columns are strings for PyTorch Forecasting
data["series"] = data["series"].astype(str)
data["static"] = data["static"].astype(str)

# define dataset
max_encoder_length = 60
max_prediction_length = 20
training_cutoff = data["time_idx"].max() - max_prediction_length

training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx= "time_idx",
    target= "value",
    group_ids=["series"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["series", "static"],
    time_varying_known_reals=[ "time_idx" ],
    time_varying_unknown_reals=[ "value" ],
    categorical_encoders={"series": NaNLabelEncoder().fit(data.series)},
)

# create validation and training dataset
validation = TimeSeriesDataSet.from_dataset(
    training,
    data,
    min_prediction_idx=training.index.time.max() + 1,
    stop_randomization=True,
)
batch_size = 128
train_dataloader = training.to_dataloader(
    train=True, batch_size=batch_size, num_workers=2
)
val_dataloader = validation.to_dataloader(
    train=False, batch_size=batch_size, num_workers=2
)

Instantiate a model using the `.from_dataset()` method.

---

In [ ]:
pl.seed_everything(42, workers=True)

deepar = DeepAR.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    rnn_layers=2,
    dropout=0.1,
    loss=NormalDistributionLoss(),
    reduce_on_plateau_patience=4,
)
print(f"Number of parameters in network: {deepar.size()/1e3:.1f}k")

Train the model with early stopping on the training dataset.

---

In [ ]:
import time

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    gradient_clip_val=0.1,
    enable_checkpointing=False,
    logger=False,
    enable_progress_bar=True,
)

start_time = time.time()

trainer.fit(deepar, train_dataloaders=train_dataloader)

print(f"Training Time: {time.time() - start_time:.2f} seconds")

Plot a sample of results.

---

In [ ]:
best_deepar = deepar

raw_predictions = best_deepar.predict(val_dataloader, mode="raw", return_x=True)

num_series = data["series"].nunique()

for idx in random.sample(range(num_series), 5):
    fig, ax = plt.subplots(figsize=(10, 5))
    best_deepar.plot_prediction(raw_predictions.x, raw_predictions.output, idx=idx, add_loss_to_title=True, ax=ax)
    plt.show()